# Natural Language Processing + Decision Science

👩🏻‍🏫 In this challenge, we will combine:
* 🗣 Natural Language Processing
* 📊 Decision Science

🎯 The goal is to understand the **negative (bad) reviews** of products and sellers on Olist.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Data Manipulation
import numpy as np
import pandas as pd
pd.set_option("display.max_columns",None)

# Machine Learning
from sklearn.pipeline import make_pipeline

# Language Processing
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import string
import unidecode as unidecode

# Vectorizers and NLP Models
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

🕵🏻‍♂️ Imagine that Olist CEO [Tiago Dalvi](https://www.linkedin.com/in/tiagodalvi/) asks you to read and understand the reviews.

- What did customers say when they rated their orders **1**, **2**, or **3** stars?
- What are the most common negative reviews?
    - About the worst-rated products?
    - About the worst-rated sellers?


## (0) Setup 🔨

First, we will load the DataFrame containing all information related to Olist reviews!

In [ ]:
df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/olist_reviews.csv")

In [ ]:
df.head()

## (2) Text Cleaning

🧹 Create a `cleaning(sentence)` function and apply it to the reviews. **Note that there is no Portuguese lemmatizer in NLTK** (usually there is not, but there is the `nltk.stem.RSLPStemmer` stemmer).

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

## (3) Analysis of Bad Reviews

### (3.1) Dataset with low review scores

😱 What proportion of reviews received a score between 1 and 3? 

In [ ]:
# YOUR CODE HERE

🕵🏻‍♂️ Let us focus on these reviews...

In [ ]:
# YOUR CODE HERE

### (3.2) Vectorization

🔡 ➡️ 🔢 Vectorize your texts.

- Make sure to account for **bigrams** (two-word expressions).
- Set `max_df = 0.75` to remove overly frequent words.
- Spoiler: you will end up with **20,000+** words…  
  For this challenge, limit to only `max_features = 5000`.

In [ ]:
# YOUR CODE HERE

### (3.3) LDA

🕵🏻‍♂️ Fit LDA:
- Choose `n_components = 3`
- Show the Document-Topic Mixture with *.transform()*
- Show the Topic Mixture with *.components_*

In [ ]:
# YOUR CODE HERE

#### Document Mixture (for Topics)

In [ ]:
# YOUR CODE HERE

👉 Let's report the most important topic for each review

In [ ]:
# YOUR CODE HERE

#### Topic Mixture (for Words)

In [ ]:
# YOUR CODE HERE

#### Topics

🎁 We provide some helper functions:
- `topic_word`: returns the most important words and their weights for a single topic
- `print_topics`: prints the different topics found by LDA with their most important words

In [ ]:
def topic_word(vectorizer, model, topic, topwords, with_weights = True):
    topwords_indexes = topic.argsort()[:-topwords - 1:-1]
    if with_weights == True:
        topwords = [(vectorizer.get_feature_names_out()[i], round(topic[i],2)) for i in topwords_indexes]
    if with_weights == False:
        topwords = [vectorizer.get_feature_names_out()[i] for i in topwords_indexes]
    return topwords

In [ ]:
def print_topics(vectorizer, model, topwords):
    for idx, topic in enumerate(model.components_):
        print("-"*20)
        print("Topic %d:" % (idx))
        print(topic_word(vectorizer, model, topic, topwords))


🕵🏻‍♂️ Print the topics with the most commonly used words:

In [ ]:
# YOUR CODE HERE

🇧🇷 Burada biraz Brezilya Portekizcesi kelimeler var:
- _cadeiras = chairs_
- _produto = product_
- _recomendo = recommend (não recomendo == not recommend)_
- _bom = good_
- _comprei = bought_
- _veio = came_
- _errado = wrong_
- _gostaria = I would like to..._
- _duas = two_
- _nao = not_
- _entregue = delivered_
- _pecas = part_
- _ainda = yet_
- _recebi = received_

👉 Show the most popular words associated with a topic

In [ ]:
# YOUR CODE HERE

In [ ]:
df["most_important_words"] = df["most_important_topic"].apply(lambda i: topic_word_mixture[i])

In [ ]:
df[["review_id",
        "review_score",
        "product_category_name",
        "full_review_cleaned",
        "most_important_topic",
        "most_important_words"]
      ].head()

## (3.4) Pipeline Tf-Idf and LDA

In [ ]:
from sklearn import set_config
set_config("diagram")

🔨 Create a Pipeline connecting the previous Vectorizer and LDA.

Fit it on the cleaned texts.

In [ ]:
# YOUR CODE HERE

💡 If you try to access the components via `pipeline.components_`, it will NOT work because Pipeline does not have `components_`. However, you can use `pipeline._final_estimator` to access the LDA. And from there you can access the topics!

In [ ]:
pipeline._final_estimator

In [ ]:
pipeline._final_estimator.components_

**Document Mixture** with Pipeline:

In [ ]:
# YOUR CODE HERE

**Topic Mixture** with Pipeline:

In [ ]:
# YOUR CODE HERE

## (4) 🎁 Product Categories

### (4.1) Grouping by product categories

📈 Group the dataset by `product_category_name` and examine their performance.

In [ ]:
# Product categories by performance - look at count, mean, median, and standard deviation
product_categories = df.groupby(by = 'product_category_name').agg({
        'review_score': ["count", "mean", "median", "std"]
    })

# Remove products with fewer sales than a threshold for analysis
cutoff = 50
product_categories = product_categories[product_categories[("review_score", "count")] > cutoff]

# Sort product categories by performance
product_categories = product_categories.sort_values(by = [('review_score', 'mean'),
                                                          ('review_score', 'std')],
                                                    ascending = [False, True])
product_categories

### (4.2) Worst product categories

👎 Save the five worst categories in terms of *average review score* into a variable called `worst_products`.

In [ ]:
worst_products = product_categories.tail(5).sort_values(by = [("review_score", "count")],
                                                       ascending = False)
worst_products

👇 Create a `worst_products_reviews` DataFrame containing only the items in `worst_products`.

In [ ]:
worst_products_reviews = df[df.product_category_name.isin(worst_products.index)]
worst_products_reviews[["review_id",
                        "review_score",
                        "product_category_name",
                        "full_review_cleaned",
                        "most_important_topic",
                        "most_important_words"]
      ]

### (4.3) Topics for the worst products

❓ What are the topics of the worst products? ❓

In [ ]:
worst_products_reviews["most_important_topic"].value_counts()

In [ ]:
bad_frequency = list(worst_products_reviews["most_important_topic"].value_counts().index)
bad_frequency

In [ ]:
[topic_word_mixture[i] for i in bad_frequency]

## (5) 🎁 Sellers...

* What kind of products were sold by the worst sellers?
* What are the main reviews for the worst sellers?

### (5.1) Worst sellers

In [ ]:
from olist.seller import Seller
sellers = Seller().get_training_data()
sellers.columns

👇 Select the 10 worst-selling sellers and save them into a variable called `worst_sellers`.

In [ ]:
worst_sellers = sellers[["seller_id", "review_score", "profits"]].sort_values(
    by = "profits",
    ascending = True).head(10)
worst_sellers

### (5.2) Products sold by the worst sellers

In [ ]:
products = Product().get_training_data() [["product_id", "category"]]
products

❓ What types of products are sold by the worst sellers? ❓

In [ ]:
sellers_product_category = data["order_items"].merge(products,
                                             on = "product_id", how = "left")[["seller_id", "category"]]

sellers_product_category

In [ ]:
sellers_product_category.groupby("seller_id").count()

### (5.3) Categories and topics for the worst sellers

🎁 Here are some useful functions:
- `focus_seller(seller_id)` to show product categories sold by a seller
- `bad_reviews_seller` to show the most popular words of the most frequent topics for a seller

In [ ]:
def focus_seller(seller_id):
    return sellers_product_category[sellers_product_category.seller_id == seller_id].value_counts()

In [ ]:
bad_reviews_sellers = worst_products_reviews.merge(data["order_items"])
bad_reviews_sellers.head(3)

In [ ]:
def bad_reviews_seller(bad_reviews_sellers, seller_id):
    mask = (bad_reviews_sellers.seller_id == seller_id)
    temp = bad_reviews_sellers[mask]
    if len(temp) > 0: # if seller appears in the bad reviews dataframe
        most_frequent_topic_seller = list(temp.most_important_topic.value_counts().head(1).index)[0]
        return topic_word_mixture[most_frequent_topic_seller]

❓ For each of these worst-selling sellers, show the most frequent product categories and words ❓

In [ ]:
for worst_seller in worst_sellers["seller_id"]:
    print("-"*50)
    print(f"Focusing on the seller #{worst_seller}...")
    print(focus_seller(worst_seller))
    print(bad_reviews_seller(bad_reviews_sellers, worst_seller))


🏁 Congratulations. You have learned some NLP basics (Preprocessing + Vectorization + NB/LDA) and combined this new "expertise" with Decision Science.

💾 Do not forget to `git add / commit / push`.